# FOLIO API Test Notebook: Item Status and Patron Holds

This notebook helps you test two FOLIO API use cases:
1. Check an item status by barcode (or item ID).
2. List patron hold requests by user ID.

## Prerequisites

Set these environment variables before running the notebook:
- `FOLIO_BASE_URL` (default: `https://api-wellcome.folio.ebsco.com`) — can override if needed
- `FOLIO_TENANT` — required
- `FOLIO_USERNAME` — required
- `FOLIO_PASSWORD` — required
- `FOLIO_CA_BUNDLE` (optional) — absolute path to your organization CA bundle if TLS interception is used
- `FOLIO_SKIP_TLS_VERIFY` (optional, `true/false`) — use `true` only for temporary debugging

In [26]:
import os
import certifi
import requests
import truststore
from pprint import pprint

# Use system certificate store (macOS Keychain, Windows, Linux trust store).
truststore.inject_into_ssl()

# Optional overrides for enterprise TLS interception or debugging.
CA_BUNDLE = os.getenv("FOLIO_CA_BUNDLE", "").strip()
SKIP_TLS_VERIFY = os.getenv("FOLIO_SKIP_TLS_VERIFY", "false").lower() == "true"

if SKIP_TLS_VERIFY:
    VERIFY_TLS = False
elif CA_BUNDLE:
    VERIFY_TLS = CA_BUNDLE
else:
    VERIFY_TLS = certifi.where()

# Set Wellcome FOLIO base URL
BASE_URL = os.getenv("FOLIO_BASE_URL", "https://api-wellcome.folio.ebsco.com").rstrip("/")
TENANT = os.getenv("FOLIO_TENANT", "")
USERNAME = os.getenv("FOLIO_USERNAME", "")
PASSWORD = os.getenv("FOLIO_PASSWORD", "")

if not all([TENANT, USERNAME, PASSWORD]):
    raise ValueError(
        "Missing required environment vars. Set FOLIO_TENANT, FOLIO_USERNAME, FOLIO_PASSWORD."
    )

print("Config loaded for tenant:", TENANT)
print("Base URL:", BASE_URL)
print("TLS verify mode:", VERIFY_TLS)

Config loaded for tenant: fs00001190
Base URL: https://api-wellcome.folio.ebsco.com
TLS verify mode: /Users/vijayvaa/wellcome_code/location-movement-poc/.venv/lib/python3.12/site-packages/certifi/cacert.pem


In [30]:
def get_okapi_token(base_url: str, tenant: str, username: str, password: str) -> str:
    login_url = f"{base_url}/authn/login"
    headers = {
        "x-okapi-tenant": tenant,
        "content-type": "application/json",
        "accept": "application/json",
    }
    payload = {"username": username, "password": password}

    response = requests.post(
        login_url,
        json=payload,
        headers=headers,
        timeout=30,
        verify=VERIFY_TLS,
    )
    if response.status_code not in (200, 201):
        raise RuntimeError(
            f"Login failed: {response.status_code} {response.text[:500]}"
        )

    token = response.headers.get("x-okapi-token")
    if not token:
        raise RuntimeError("Login succeeded but x-okapi-token header was missing.")

    return token


TOKEN = get_okapi_token(BASE_URL, TENANT, USERNAME, PASSWORD)
print("Authenticated. Token received.")

Authenticated. Token received.


In [32]:
def folio_get(path: str, params: dict | None = None) -> dict:
    global TOKEN
    url = f"{BASE_URL}{path}"

    def _headers() -> dict:
        return {
            "x-okapi-tenant": TENANT,
            "x-okapi-token": TOKEN,
            "accept": "application/json",
        }

    response = requests.get(
        url,
        headers=_headers(),
        params=params or {},
        timeout=30,
        verify=VERIFY_TLS,
    )

    # If token expired, refresh once and retry the request.
    if response.status_code == 401:
        TOKEN = get_okapi_token(BASE_URL, TENANT, USERNAME, PASSWORD)
        response = requests.get(
            url,
            headers=_headers(),
            params=params or {},
            timeout=30,
            verify=VERIFY_TLS,
        )

    if response.status_code >= 400:
        raise RuntimeError(
            f"GET {path} failed: {response.status_code} {response.text[:500]}"
        )

    return response.json()

## 1) Check Item Status

Use a barcode to find the item, then print status details.

In [44]:
ITEM_ID = "it00002975182"

query = f'hrid="{ITEM_ID}"'
items_result = folio_get("/inventory/items", params={"query": query, "limit": 1})
items = items_result.get("items", [])

if not items:
    print(f"No item found for id: {ITEM_ID}")
else:
    item = items[0]
    print("Item Found:")
    print("- id:", item.get("id"))
    print("- barcode:", item.get("barcode"))
    print("- status:", (item.get("status") or {}).get("name"))
    print("- materialTypeId:", item.get("materialTypeId"))
    print("- permanentLoanTypeId:", item.get("permanentLoanTypeId"))
    print("- temporaryLoanTypeId:", item.get("temporaryLoanTypeId"))
    print("\nRaw item payload:")
    pprint(item)

Item Found:
- id: fdfb9bb4-e4a8-5978-8e60-5f5523ff2e05
- barcode: X96956
- status: Available
- materialTypeId: None
- permanentLoanTypeId: None
- temporaryLoanTypeId: None

Raw item payload:
{'_version': '2',
 'administrativeNotes': [],
 'barcode': 'X96956',
 'callNumber': 'EPB/D/2034',
 'circulationNotes': [{'date': None,
                       'id': '6fc06f78-95c5-4fa8-85dc-af3a510285a2',
                       'note': 'default',
                       'noteType': 'Check in',
                       'source': {'id': '06b5d70d-03dd-4f5d-bda8-2626c65e8c6d',
                                  'personal': {'firstName': 'Migration',
                                               'lastName': 'Data'}},
                       'staffOnly': True}],
 'contributorNames': [],
 'copyNumber': '1',
 'discoverySuppress': False,
 'effectiveCallNumberComponents': {'callNumber': 'EPB/D/2034',
                                   'prefix': None,
                                   'suffix': None,
            

In [42]:
# Item notes on an inventory item are usually in `notes` (typed notes) and sometimes `administrativeNotes`.
notes = (item or {}).get("notes", [])
admin_notes = (item or {}).get("administrativeNotes", [])

print(f"Typed item notes count: {len(notes)}")
for idx, note_obj in enumerate(notes, start=1):
    print(f"\n{idx}. note:", note_obj.get("note"))
    print("   itemNoteTypeId:", note_obj.get("itemNoteTypeId"))
    print("   staffOnly:", note_obj.get("staffOnly"))

print(f"\nAdministrative notes count: {len(admin_notes)}")
for idx, note_text in enumerate(admin_notes, start=1):
    print(f"{idx}. {note_text}")

Typed item notes count: 25

1. note: f
   itemNoteTypeId: f4d07dfc-ced4-47ed-b760-39b77b6bf457
   staffOnly: True

2. note: Inventory, found: Q2 11/02/2026: Alexandra Hill
   itemNoteTypeId: 8d0a5eca-25de-4391-81a9-236eeefdd20b
   staffOnly: False

3. note: 0
   itemNoteTypeId: 967aee80-37f4-4ff5-8c9a-151f380c288f
   staffOnly: True

4. note:   -  -  
   itemNoteTypeId: f027396e-268b-4f0a-8913-9075bfe0a154
   staffOnly: True

5. note: 0
   itemNoteTypeId: e34b1a3f-66c9-40e9-8840-729b6d5d22f8
   staffOnly: True

6. note: 0
   itemNoteTypeId: 620a7bb5-436e-4a6a-b375-757e120948aa
   staffOnly: True

7. note:   -  -  
   itemNoteTypeId: de2225e3-5769-459c-9f06-087ff155bb0c
   staffOnly: True

8. note: 0
   itemNoteTypeId: 80becb2f-a7bb-46c1-9a11-12e98feb4f0f
   staffOnly: True

9. note: 0
   itemNoteTypeId: 8107a516-db58-46e3-b483-b882f3884c07
   staffOnly: True

10. note: 0
   itemNoteTypeId: d9c15b26-951c-4ab9-94ff-08a34a82da21
   staffOnly: True

11. note:   -  -  
   itemNoteTypeId: ff

In [43]:
# Resolve itemNoteTypeId -> note type name for easier reading.
def load_item_note_type_map() -> dict[str, str]:
    endpoints = ["/item-note-types", "/inventory/item-note-types"]
    last_error = None

    for endpoint in endpoints:
        try:
            payload = folio_get(endpoint, params={"limit": 2000})
            note_types = payload.get("itemNoteTypes", [])
            if note_types:
                return {nt.get("id"): nt.get("name", "<unnamed>") for nt in note_types if nt.get("id")}
        except Exception as exc:
            last_error = exc

    if last_error:
        print("Could not load item note types:", last_error)
    return {}


note_type_map = load_item_note_type_map()
print(f"Loaded note type mappings: {len(note_type_map)}")

notes = (item or {}).get("notes", [])
for idx, note_obj in enumerate(notes, start=1):
    note_type_id = note_obj.get("itemNoteTypeId")
    note_type_name = note_type_map.get(note_type_id, "<unknown type>")
    print(f"{idx}. type={note_type_name} ({note_type_id}) | staffOnly={note_obj.get('staffOnly')}")
    print(f"   note={note_obj.get('note')}")

Loaded note type mappings: 46
1. type=OPACMSG (f4d07dfc-ced4-47ed-b760-39b77b6bf457) | staffOnly=True
   note=f
2. type=Note (8d0a5eca-25de-4391-81a9-236eeefdd20b) | staffOnly=False
   note=Inventory, found: Q2 11/02/2026: Alexandra Hill
3. type=Last Patron-Sierra (967aee80-37f4-4ff5-8c9a-151f380c288f) | staffOnly=True
   note=0
4. type=Inventory Date-Sierra (f027396e-268b-4f0a-8913-9075bfe0a154) | staffOnly=True
   note=  -  -  
5. type=Num of Renewals-Sierra (e34b1a3f-66c9-40e9-8840-729b6d5d22f8) | staffOnly=True
   note=0
6. type=Num of Overdues-Sierra (620a7bb5-436e-4a6a-b375-757e120948aa) | staffOnly=True
   note=0
7. type=Last Overdue Date-Sierra (de2225e3-5769-459c-9f06-087ff155bb0c) | staffOnly=True
   note=  -  -  
8. type=IUSE3 (80becb2f-a7bb-46c1-9a11-12e98feb4f0f) | staffOnly=True
   note=0
9. type=Total Checkouts-Sierra (8107a516-db58-46e3-b483-b882f3884c07) | staffOnly=True
   note=0
10. type=Total Renewals-Sierra (d9c15b26-951c-4ab9-94ff-08a34a82da21) | staffOnly=True
  

## 2) List Patron Holds

Use a FOLIO user ID (UUID) to list hold requests (`requestType=="Hold"`).

In [ ]:
PATRON_USER_ID = "REPLACE_WITH_PATRON_USER_UUID"

holds_query = f'(requesterId=={PATRON_USER_ID} and requestType=="Hold") sortby requestDate/sort.descending'
holds_result = folio_get("/circulation/requests", params={"query": holds_query, "limit": 100})
requests_list = holds_result.get("requests", [])

print(f"Total hold requests: {len(requests_list)}")
for idx, req in enumerate(requests_list, start=1):
    print(f"\n{idx}. Request ID: {req.get('id')}")
    print("   Status:", req.get("status"))
    print("   Position:", req.get("position"))
    print("   Request Date:", req.get("requestDate"))
    print("   Item ID:", req.get("itemId"))
    print("   Instance ID:", req.get("instanceId"))

print("\nRaw response payload:")
pprint(holds_result)

## Optional: Filter Active Holds Only

If you only want active/open hold statuses, adjust the query in Cell 9.
Common statuses may include values like `Open - Not yet filled` or `Open - Awaiting pickup` depending on your tenant configuration.